In [1]:
from biASE import General_Bias_Calculator
from ase.atoms import Atoms
from ase import units
from ase.io import read


from ase.md.langevin import Langevin
from ase.md.verlet import VelocityVerlet
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution,Stationary, ZeroRotation
from ase.constraints import FixedPlane


import numpy as np
import matplotlib.pyplot as plt
from ase.calculators.calculator import Calculator, all_changes
from ase.io import Trajectory
import copy

from ase.calculators.mixing import LinearCombinationCalculator,MixedCalculator
import time
import jax.numpy as jnp
from jax import jit
from jax import grad ,jacfwd,jacrev
from ase import Atoms
from Metadynamics import Metadynamics
from Metadynamics import WT_Metadynamics
import cmocean.cm as cmo

from ase import units
from ase.io import read
from ase.constraints import FixedPlane
from ase.calculators.lj import LennardJones

def plot_2d_bias(f_bias,rangeCV1=[-1,1],rangeCV2=[-1,1],N_points=100,trajectory=None,pic_name = '2D_plot.png'):
    x = np.linspace(rangeCV1[0], rangeCV1[1], N_points)
    y = np.linspace(rangeCV2[0], rangeCV2[1], N_points)

    X, Y = np.meshgrid(x, y)
    CVs=[X,Y]
    Z = f_bias(CVs)
    
    plt.contour(X, Y, Z, 50, cmap=cmo.tempo_r)#'RdGy')
    plt.colorbar()
    
    if trajectory==None:
        plt.show()
    else:
        plt.plot(trajectory[0],trajectory[1])
        plt.xlabel('x')
        plt.ylabel('y')
        plt.savefig(pic_name)
        plt.show()

def plot_1d_bias(f_bias,rangeCV1=[-0.1,1.5],N_points=100,trajectory=None,pic_name = '1D_DW_C.png'):
    x = np.linspace(rangeCV1[0], rangeCV1[1], N_points)
    
    try:
        CVs=[x] #needs to be an array for the spring method
        Z = f_bias(CVs)
        plt.plot(x, Z)
    except:
        CVs=np.array(x) #needs to be an array for the spring method
        Z = f_bias(CVs)
        plt.plot(x, Z)
    if trajectory==None:
        plt.show()
    else:
        plt.hist(trajectory[0],density=True,bins=100,range=rangeCV1)
        plt.xlabel('x')
        plt.ylabel('E')
        plt.savefig(pic_name)
        plt.show()
        

        
def _cvs(atom_pos):
    '''Collective variable from the x cooridinate'''
    x = atom_pos[0][0]
    return jnp.array([x])

def _cvs_2D(atom_pos):
    '''Collective variable from the x and y cooridinate'''
    x = atom_pos[0][0]
    y = atom_pos[0][1]
    return jnp.array([x,y])
"""
def _cvs(atom_pos):
    '''Collective variable from the x, y and z cooridinate'''
    x = atom_pos[0][0]
    y = atom_pos[0][1]
    z = atom_pos[0][2]
    return jnp.array([x,y,z])"""

def f_dw_cvs(cvs,scale=0.2):
    '''Double-Well'''
    x =  cvs[0]
    y =  cvs[1]
    return scale*((x**2 - 1)**2 + y**2)

def f_cvs(cvs,scale=.04):
    x = cvs[0]
    y = cvs[1]
    r = jnp.sqrt(x**2 + y**2)
    large_min = jnp.exp(-0.5*((r)/0.5)**2)
    small_mins = 0
    for i in range(5):
        x0 = 1*jnp.cos(2*jnp.pi*i/5)
        y0 = 1*jnp.sin(2*jnp.pi*i/5)
        small_mins += 1*jnp.exp(-0.5*((x-x0)**2 + (y-y0)**2)/0.2**2)

        x1 = 1.5*jnp.cos(2*jnp.pi*i/5 + jnp.pi/5)
        y1 = 1.5*jnp.sin(2*jnp.pi*i/5 + jnp.pi/5)
        small_mins += 0.8*jnp.exp(-0.2*((x-x1)**2 + (y-y1)**2)/0.2**2)

        x2=4.5*jnp.cos(2*jnp.pi*i/5 + jnp.pi/(2.5))
        y2= 4.5*jnp.sin(2*jnp.pi*i/5 + jnp.pi/(2.5))
        small_mins += 1*jnp.exp(-0.1*((x-x2)**2 + (y-y2)**2)/0.2**2)

    return scale*(-10*large_min - 10*small_mins + (x**2 +y**2))

def f_cvs_1D(cvs):
    x = cvs[0]
    return x**2*(1.-x)**2

def deriv_f_cvs_1D(cvs):
    x = cvs[0]
    df_dx = 2*x*(1.-x)*(2*x-1.)
    return np.array([[df_dx,0,0]])

def deriv_f_cvs_1D(cvs, h=1e-4):
    f0 = f_cvs_1D(cvs)
    x = cvs[0]
    fx = f_cvs_1D([x+h])
    fd = (fx - f0) / h
    return np.array([[fd,0,0]])

In [2]:
import numpy as np
from scipy.stats import moment



import numpy as np
from scipy.stats import moment
from ase import Atoms
from ase.build import molecule
from ase.neighborlist import neighbor_list

def get_coordination_numbers(atoms_positions, cutoff):
    dist_matrix = np.sqrt(np.sum((atoms_positions[:, np.newaxis] - atoms_positions) ** 2, axis=-1))
    x_matrix = (1 - (dist_matrix / cutoff) ** 8) / (1 - (dist_matrix / cutoff) ** 16)
    print(x_matrix)
    np.fill_diagonal(x_matrix, 0)  # Set diagonal elements to zero
    coordination_numbers = np.sum(x_matrix, axis=1)
    return coordination_numbers


def get_coordination_moments_numpy(coord_nums):
    """Calculate the second and third central moments of the coordination number distribution using numpy."""
    mean = np.mean(coord_nums)
    scm = np.sum((coord_nums - mean)**2) / len(coord_nums)
    tcm = np.sum((coord_nums - mean)**3) / len(coord_nums)
    return scm, tcm


def fill_diagonal(matrix, value):
    num_rows, num_cols = matrix.shape
    diagonal_indices = jnp.arange(min(num_rows, num_cols))
    filled_matrix = jax.ops.index_update(matrix, (diagonal_indices, diagonal_indices), value)
    return filled_matrix

def _cvs_moments(atoms_positions, cutoff):
    """ Calculate coordination numbers for all atoms in an ASE Atoms object.
        And then calculate the second and third central moments of the coordination number distribution using numpy.
    """
    dist_matrix = jnp.sqrt(jnp.sum((atoms_positions[:, jnp.newaxis] - atoms_positions) ** 2, axis=-1))
    x_matrix = (1 - (dist_matrix / cutoff) ** 8) / (1 - (dist_matrix / cutoff) ** 16)
    #print(x_matrix.shape[0])
    x_matrix = x_matrix.at[jnp.diag_indices(x_matrix.shape[0])].set(0) # there is no jnp.fill_diagonal(x_matrix, 0)  # Set diagonal elements to zero
    coord_nums = jnp.sum(x_matrix, axis=1)
    #print(coord_nums)
    mean = jnp.mean(coord_nums)
    scm = jnp.sum((coord_nums - mean)**2) / len(coord_nums)
    tcm = jnp.sum((coord_nums - mean)**3) / len(coord_nums)
    #print(scm, tcm)
    return scm, tcm





In [3]:
"""timestep = 0.005
#.5 * units.fs
atoms = read('isomer.xyz')
print(_cvs_moments(atoms.get_positions(), cutoff=1.5))
print(atoms.get_velocities())
cons = [FixedPlane(i, [0, 0, 1]) for i in range(7)]
atoms.set_constraint(cons)
atoms.set_masses([1, 1, 1, 1, 1, 1, 1])
atoms.calc = LennardJones(rc=2.5, r0=3.0)
'''calc1= LennardJones(rc=2.5, r0=3.0)
calc2 = LennardJones(rc=2.5, r0=3.5)
atoms.calc = LinearCombinationCalculator([calc1,calc2],[0.5,0.5])'''
from functools import partial
dyn = WT_Metadynamics(atoms, timestep=timestep, temperature_K=0.1/units.kB, friction=1, trajectory='metad3.traj'
                      ,fixcm=False,cvs=partial(_cvs_moments, cutoff=1.5),interval_size=500,std_dev=0.1,bias_height=.05,output_file='test3.txt',well_temp=0.1) #Langevin(atoms, timestep=.5 * units.fs, temperature_K=0.1/units.kB, friction=1, trajectory='metad.traj',fixcm=False) #
N_steps = int(1e5)
dyn.run(N_steps) """

"timestep = 0.005\n#.5 * units.fs\natoms = read('isomer.xyz')\nprint(_cvs_moments(atoms.get_positions(), cutoff=1.5))\nprint(atoms.get_velocities())\ncons = [FixedPlane(i, [0, 0, 1]) for i in range(7)]\natoms.set_constraint(cons)\natoms.set_masses([1, 1, 1, 1, 1, 1, 1])\natoms.calc = LennardJones(rc=2.5, r0=3.0)\n'''calc1= LennardJones(rc=2.5, r0=3.0)\ncalc2 = LennardJones(rc=2.5, r0=3.5)\natoms.calc = LinearCombinationCalculator([calc1,calc2],[0.5,0.5])'''\nfrom functools import partial\ndyn = WT_Metadynamics(atoms, timestep=timestep, temperature_K=0.1/units.kB, friction=1, trajectory='metad3.traj'\n                      ,fixcm=False,cvs=partial(_cvs_moments, cutoff=1.5),interval_size=500,std_dev=0.1,bias_height=.05,output_file='test3.txt',well_temp=0.1) #Langevin(atoms, timestep=.5 * units.fs, temperature_K=0.1/units.kB, friction=1, trajectory='metad.traj',fixcm=False) #\nN_steps = int(1e5)\ndyn.run(N_steps) "

In [4]:
def gaussian_1D(cvs,std_dev,mean_cvs,bias_height,regulation_factor): ### here with jax numpy
        """ 
        x (float or numpy array): Input value(s) for evaluating the Gaussian.
            mean (float or numpy array): Mean(s) of the Gaussian(s).
            std_dev (float): Standard deviation of the Gaussian(s).

        Returns:
            float or numpy array: Value(s) of the Gaussian(s) at the given input(s), summed over all means.
            
            
            this function needs to be tested! seems to be the same as the above numpy version but forces need to be tested
        """ 
        x = cvs[0]

        # Evaluate the Gaussian for all means at once using numpy broadcasting
        gaussian_vals = (jnp.exp(-0.5*((x - mean_cvs[:, jnp.newaxis])**2/(std_dev**2)))).flatten()
        reg_f =regulation_factor.flatten()
        #print('regulation_factor: ', self.previous_portential,self.nsteps)
        # Sum over all means and return the result
        #print(jnp.sum(gaussian_vals, axis=0)[0])
        #print('Previous Potential: ',  self.bias_height*self.regulation_factor*jnp.sum(gaussian_vals, axis=0)[0],self.nsteps)
        '''print('gauss: ',gaussian_vals)
        print('reg: ' ,reg_f)
        print('gauss times reg',gaussian_vals*reg_f)
        print('sum of gauss times reg ',jnp.sum(gaussian_vals*reg_f, axis=0))'''
        return bias_height*jnp.sum(gaussian_vals*reg_f, axis=0)

In [5]:
def gaussian_1D_np(cvs,std_dev,mean_cvs,bias_height,regulation_factor): ### here with jax numpy
        """ 
        x (float or numpy array): Input value(s) for evaluating the Gaussian.
            mean (float or numpy array): Mean(s) of the Gaussian(s).
            std_dev (float): Standard deviation of the Gaussian(s).

        Returns:
            float or numpy array: Value(s) of the Gaussian(s) at the given input(s), summed over all means.
            
            
            this function needs to be tested! seems to be the same as the above numpy version but forces need to be tested
        """ 
        
        x = cvs[0]

        # Evaluate the Gaussian for all means at once using numpy broadcasting
        gaussian_vals = (np.exp(-0.5*((x - mean_cvs[:, np.newaxis])**2/(std_dev**2)))).flatten()
        reg_f =regulation_factor.flatten()
        #print('regulation_factor: ', self.previous_portential,self.nsteps)
        # Sum over all means and return the result
        #print(jnp.sum(gaussian_vals, axis=0)[0])
        #print('Previous Potential: ',  self.bias_height*self.regulation_factor*jnp.sum(gaussian_vals, axis=0)[0],self.nsteps)
        '''print('gauss: ',gaussian_vals)
        print('reg: ' ,reg_f)
        print('gauss times reg',gaussian_vals*reg_f)
        print('sum of gauss times reg ',np.sum(gaussian_vals*reg_f, axis=0))'''
        return bias_height*np.sum(gaussian_vals*reg_f, axis=0)

In [6]:
# Initialize lists to store the columns
time_list = []
d1_list = []
sigma_d1_list = []
height_list = []
biasf_list = []

# Specify the path to your .txt file
file_path = "DIMER_long_pace"

# Read the file line by line and extract data
with open(file_path, "r") as file:
    for line in file:
        if not line.startswith("#"):  # Skip lines starting with "#"
            columns = line.split()
            # Convert values to appropriate data types and append to respective lists
            
            d1_list.append(float(columns[1]))
            height_list.append(float(columns[3]))

'''
# Print the lists to verify the data

print("d1:", d1_list)
print("height:", height_list)'''


'\n# Print the lists to verify the data\n\nprint("d1:", d1_list)\nprint("height:", height_list)'

In [7]:
regulation_factor1 = np.array([[1]])
bias_height1 = 0.125
std_dev1 = 0.2
bias_factor=5
well_temp=0.05
q_list= []
for i in range(len(d1_list[:5])):
    if i ==0:
        print("My result/ PLUMED result",gaussian_1D(np.array([d1_list[i]]),std_dev1,np.array([d1_list[i]]),bias_height1,regulation_factor1)/height_list[i],"\t \t \t",regulation_factor1[-1])
    else:
        mean_cvs1 = np.array([d1_list[:i]])
        print("m: ",mean_cvs1)
        cvs1=np.array([d1_list[i]])
        print("c: ",cvs1)
        previous_portential = gaussian_1D(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1)
        #print(previous_portential)

        regulation_factor1 = np.vstack([regulation_factor1, np.exp(-previous_portential/(well_temp*bias_factor))] )
        #print(regulation_factor1)
        #print('output in file:')
        mean_cvs1 = np.array([d1_list[i]])
        cvs1=np.array([d1_list[i]])
        #print(gaussian_1D(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1[-1]))
        if i%10==0:
            print("My result/ PLUMED result", gaussian_1D(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1[-1])/height_list[i],"\t",regulation_factor1[-1])
        q_list.append(gaussian_1D(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1[-1])/height_list[i])

My result/ PLUMED result 1.0 	 	 	 [1]
m:  [[2.63228964]]
c:  [2.63559691]
m:  [[2.63228964 2.63559691]]
c:  [2.63853313]
m:  [[2.63228964 2.63559691 2.63853313]]
c:  [2.64045441]
m:  [[2.63228964 2.63559691 2.63853313 2.64045441]]
c:  [2.64436261]


In [8]:
print(max(q_list),min(q_list))

0.9999998 0.9999978


In [9]:
regulation_factor1 = np.array([[1]])
bias_height1 = 0.125
std_dev1 = .2
print("My result/ PLUMED result",gaussian_1D_np(np.array([2.434683333 ]),std_dev1,np.array([2.633582910836874]),bias_height1,regulation_factor1))
 

My result/ PLUMED result 0.0762334781116698


In [10]:
0.012209002/0.01220900189959116

1.0000000082241645

In [11]:
0.01220900189959116/0.011990843

1.0181937916784634

In [12]:
#kbt_*(biasf_-1.0))/height0_*getTimeStep()*stride_
def calculate_tau(kbt, biasf, height0, time_step, stride):
    tau = (kbt * (biasf - 1.0)) / (height0 * time_step * stride)
    return tau
print(0.05*(5-1)*(0.002*100)/(0.1))

0.4000000000000001


In [13]:
def gaussian_2D_WT(cvs,std_dev,mean_cvs,bias_height,regulation_factor):
    """
    Evaluate a 2D Gaussian potential energy surface for the given input collective variable(s) (cvs).

    Args:
        cvs (float or numpy array): Input value(s) for evaluating the 2D Gaussian.

    Returns:
        float or numpy array: Value(s) of the 2D Gaussian at the given input(s), summed over all means.
    """
    x, y = cvs
    #print(mean_cvs[:, 0])
    # Evaluate the Gaussian for all means at once using numpy broadcasting
    gaussian_vals = jnp.exp(-0.5*((x - mean_cvs[:, 0])**2/(std_dev**2)) \
                                                    - 0.5*((y - mean_cvs[:, 1])**2/(std_dev**2)))
    #regulation_factor=jnp.exp(-self.previous_portential/(self.temp*self.bias_factor)) #the kB here is used to transform from K to eV!
    #print('regulation_factor: ', self.previous_portential,self.nsteps)
    #print(mean_cvs,cvs)
    # Sum over all means and return the result
    #print(jnp.sum(gaussian_vals, axis=0)[0])
    
    '''print('Potential_NOW: ', self.bias_height*regulation_factor*jnp.sum(gaussian_vals,axis=0),self.nsteps)
    print('Sum_Gaussian:', jnp.sum(gaussian_vals,axis=0), 'Previous Potential: ', self.previous_portential,self.nsteps)
    print('Reg_factor: ',regulation_factor, 'T: ', self.temp)'''
    #print('HI')
    #print(self.regulation_factor)
    reg_f =regulation_factor.flatten()
    ''' print('gauss: ',gaussian_vals)
    print('reg: ' ,reg_f)
    print('gauss times reg',gaussian_vals*reg_f)
    print('sum of gauss times reg ',jnp.sum(gaussian_vals*reg_f, axis=0))'''
    return bias_height*jnp.sum(gaussian_vals*reg_f,axis=0)

In [14]:
# Initialize lists to store the columns
time_list = []
height_list = []
sigma_d1_list = []
scm_list = []
tcm_list = []
biasf_list = []

# Specify the path to your .txt file
file_path = "HILLS"

# Read the file line by line and extract data
with open(file_path, "r") as file:
    for line in file:
        if not line.startswith("#"):  # Skip lines starting with "#"
            columns = line.split()
            # Convert values to appropriate data types and append to respective lists
            
            scm_list.append(float(columns[1]))
            tcm_list.append(float(columns[2]))
            height_list.append(float(columns[5]))


In [15]:
# Combine the two lists into a nx2 NumPy array.
scm_tcm = np.array(list(zip(scm_list, tcm_list)))


In [16]:
a=np.array([scm_tcm[:,:2]])
print(a)
print(a[:,1])

[[[0.74668349 1.31008521]
  [0.74694509 1.27041599]
  [0.74240222 1.29654323]
  ...
  [0.73983406 1.29094168]
  [0.70936215 1.17444746]
  [0.6904114  1.10669043]]]
[[0.74694509 1.27041599]]


In [19]:
regulation_factor1 = np.array([[1]])
bias_height1 = 0.0625
std_dev1 = 0.1
bias_factor=5
well_temp=0.1
q_list= []
for i in range(len(height_list)):
    if i ==0:
        print("My result/ PLUMED result",gaussian_2D_WT(scm_tcm[i],std_dev1,scm_tcm[:1],bias_height1,regulation_factor1)/height_list[i],"\t \t \t",regulation_factor1[-1])
    else:
        mean_cvs1 = scm_tcm[:i]
        #print("m: ",mean_cvs1) 
        cvs1=scm_tcm[i]
        #print("c: ",cvs1)
        previous_portential = gaussian_2D_WT(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1)
        #print(previous_portential)

        regulation_factor1 = np.vstack([regulation_factor1, np.exp(-previous_portential/(well_temp*bias_factor))] )
        #print(regulation_factor1)
        #print('output in file:')
        mean_cvs1 = np.array([scm_tcm[i]])
        cvs1=scm_tcm[i]
        #print(gaussian_1D(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1[-1]))
        if i%10==0:
            print(height_list[i])
            print("My result/ PLUMED result", gaussian_2D_WT(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1[-1])/height_list[i],"\t",regulation_factor1[-1])
        q_list.append(gaussian_2D_WT(cvs1,std_dev1,mean_cvs1,bias_height1,regulation_factor1[-1])/height_list[i])

My result/ PLUMED result 1.0 	 	 	 [1]
0.02843561816023077
My result/ PLUMED result 0.99989367 	 [0.45492151]
0.01925681074924064
My result/ PLUMED result 0.9996748 	 [0.30800879]
0.01542146557959716
My result/ PLUMED result 0.9994727 	 [0.24661334]
0.0127609864745152
My result/ PLUMED result 0.9992755 	 [0.20402786]
0.01562813823102876
My result/ PLUMED result 0.9987297 	 [0.24973258]
0.05567785273476147
My result/ PLUMED result 0.9990547 	 [0.8900035]
0.009001492332754747
My result/ PLUMED result 0.99900895 	 [0.14388114]
0.03775246789820852
My result/ PLUMED result 0.99872065 	 [0.60326672]
0.02882763611084718
My result/ PLUMED result 0.99629325 	 [0.45953247]
0.02765696219752565
My result/ PLUMED result 0.9984095 	 [0.4418076]
0.03127644610113263
My result/ PLUMED result 0.9996928 	 [0.50026941]
0.02102202514524629
My result/ PLUMED result 0.9993881 	 [0.33614659]
0.0367155883790837
My result/ PLUMED result 0.9979268 	 [0.58623153]
0.006704022678460085
My result/ PLUMED result 0.99

In [20]:
print(max(q_list),min(q_list))

1.0 0.9860823
